In [ ]:
!pip -q install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.


In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — MONDAY (Improved)
# UPDATED: runtime + simple complexity reporting
# - Measures: total wall time, solve time, total runtime
# - Prints: problem size (nodes, vehicles), approx complexity terms
# ============================================================

import glob
import math
import time
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

t_total0 = time.perf_counter()   # TOTAL runtime start

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

DAY = 24 * 60

VEHICLE_FIXED_COST = 250000
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"{hh:02d}:{mm:02d}"

def route_miles(route_nodes, dist_miles):
    m = 0.0
    for a, b in zip(route_nodes, route_nodes[1:]):
        m += dist_miles[a][b]
    return m

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

monday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("monday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("monday",))
)
if monday_sheet is None:
    raise ValueError("Could not find a Monday sheet (e.g., 'Monday Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))

print("Distance sheet:", distance_sheet)
print("Monday sheet  :", monday_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN MONDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=monday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Monday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA")
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

# --- Complexity (rough) ---
# OR-Tools VRP with metaheuristics is not polynomial; worst-case grows super-exponentially.
# We report a *practical* scale proxy:
#   callbacks build: O(n^2)
#   routing search: bounded by time_limit; each iteration evaluates neighborhoods ~ O(k * n) (heuristic)
# These are approximations for reporting only.
n2 = n * n
print("\n--- Complexity proxy (reporting) ---")
print(f"n nodes (incl. DC): {n}")
print(f"vehicles available : {num_vehicles}")
print(f"distance matrix build ~ O(n^2) = {n2:,} pair-evals")
print(f"local search bounded by time_limit = {SOLVER_SECONDS}s (heuristic / NP-hard)")
print("-----------------------------------\n")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)

routing.AddDimension(
    time_idx,
    24 * 60,
    24 * 60,
    False,
    "Time"
)
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.use_full_propagation = True
params.log_search = True
params.time_limit.FromSeconds(SOLVER_SECONDS)

t_solve0 = time.perf_counter()
solution = routing.SolveWithParameters(params)
t_solve1 = time.perf_counter()

solve_seconds = t_solve1 - t_solve0
print(f"\n--- Runtime ---")
print(f"OR-Tools solve time: {solve_seconds:.3f} seconds (time_limit={SOLVER_SECONDS}s)")
print("----------------\n")

if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "All stops cannot be served 08:00–18:00 with given capacity.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "monday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

t_total1 = time.perf_counter()

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")

print("\n--- Total runtime ---")
print(f"End-to-end runtime: {(t_total1 - t_total0):.3f} seconds")
print("---------------------")

Using file: solution deliveries Monday (1).xlsx
Sheets: ['Sheet1', 'OrderTable', 'monday pivot', 'Monday Final', 'Monday', 'distance', 'LocationTable']
Distance sheet: distance
Monday sheet  : Monday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Monday stores (original rows): 34
Total cube: 10223.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 34
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 44

--- Complexity proxy (reporting) ---
n nodes (incl. DC): 35
vehicles available : 44
distance matrix build ~ O(n^2) = 1,225 pair-evals
local search bounded by time_limit = 180s (heuristic / NP-hard)
-----------------------------------



/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



--- Runtime ---
OR-Tools solve time: 180.006 seconds (time_limit=180s)
----------------

Routes from OR-Tools (before HOS split): 4
Routes after HOS enforcement: 4

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1     10     2921.0       421.0          D0 04:58          D1 06:30              04:58              30:30              600         10.53        15.53
     2     12     3199.0       300.0          D0 06:56          D0 20:40              06:56              20:40                0          7.50        13.74
     3      3     2365.0       507.0          D0 07:36          D1 08:10              07:36              32:10              600         12.68        14.57
     4      9     1738.0        73.0          D0 07:48          D0 14:08              07:48              14:08                0          1.82         6.33

Saved: monday_ortools_ro

In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — MONDAY (Improved)
# - Better first solution + local search
# - Adds fixed vehicle cost to reduce # of trucks
# - Optional soft cluster grouping (if LocationTable has Cluster_Hierarchical)
# - Same output: Summary / Schedule / TruckPlan
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180      # give it a bit more time for better routes
EXTRA_VEHICLES = 40       # allow enough vehicles but penalize usage via fixed cost

DAY = 24 * 60

# Objective shaping:
VEHICLE_FIXED_COST = 250000  # higher => fewer trucks (tune 50k–500k)
DIST_COST_SCALE    = 1000    # keep your arc cost scale
CLUSTER_PENALTY    = 2000    # soft penalty for switching clusters (tune 0–10000)

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

monday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("monday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("monday",))
)
if monday_sheet is None:
    raise ValueError("Could not find a Monday sheet (e.g., 'Monday Final').")

# Optional location table (for soft cluster preference)
location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., "LocationTable"

print("Distance sheet:", distance_sheet)
print("Monday sheet  :", monday_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN MONDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=monday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Monday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    # try common names
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA")
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

# Arc cost: distance + optional cluster switching penalty
def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    # soft preference: avoid switching clusters often
    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

# Encourage fewer vehicles
for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

# Time callback: travel + service at FROM node (standard)
def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)

# Give slack so waiting is allowed; horizon 24h is enough for in-day deliveries
routing.AddDimension(
    time_idx,
    24 * 60,   # slack (waiting)
    24 * 60,   # horizon
    False,
    "Time"
)
time_dim = routing.GetDimensionOrDie("Time")

# Store time windows (arrival must allow service to finish by 18:00)
for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

# Depot can start anytime up to 08:00 (dispatch will be computed later)
for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

# Search parameters (much better quality typically)
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH

# strong neighborhood moves
params.use_full_propagation = True
params.log_search = True  # set False if you want quieter output
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "All stops cannot be served 08:00–18:00 with given capacity.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input."
    )

# Extract non-empty routes
routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    # Return to DC
    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "monday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Monday (1).xlsx
Sheets: ['Sheet1', 'OrderTable', 'monday pivot', 'Monday Final', 'Monday', 'distance', 'LocationTable']
Distance sheet: distance
Monday sheet  : Monday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Monday stores (original rows): 34
Total cube: 10223.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 34
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 44


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 4
Routes after HOS enforcement: 4

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1     10     2921.0       421.0          D0 04:58          D1 06:30              04:58              30:30              600         10.53        15.53
     2     12     3199.0       300.0          D0 06:56          D0 20:40              06:56              20:40                0          7.50        13.74
     3      3     2365.0       507.0          D0 07:36          D1 08:10              07:36              32:10              600         12.68        14.57
     4      9     1738.0        73.0          D0 07:48          D0 14:08              07:48              14:08                0          1.82         6.33

Saved: monday_ortools_routes_schedule.xlsx
Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day
HH:MM>24 colu

Tuesday

In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — TUESDAY (Improved)
#
# Improvements vs basic version:
# - Auto-picks Tuesday sheet (e.g., "Tuesday Final", "Tuesday", etc.)
# - Better first solution: PARALLEL_CHEAPEST_INSERTION
# - Adds fixed vehicle cost to reduce # of trucks
# - Optional soft cluster grouping if LocationTable has Cluster_Hierarchical
# - Output: tuesday_ortools_routes_schedule.xlsx (Summary / Schedule / TruckPlan)
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

DAY = 24 * 60

# Objective shaping:
VEHICLE_FIXED_COST = 250000   # higher => fewer trucks (tune 50k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000     # set 0 to disable

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

tuesday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("tuesday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("tuesday",))
)
if tuesday_sheet is None:
    raise ValueError("Could not find a Tuesday sheet (e.g., 'Tuesday Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet:", distance_sheet)
print("Tuesday sheet :", tuesday_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN TUESDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=tuesday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)

if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Tuesday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.use_full_propagation = True
params.log_search = True
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "tuesday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Tuesday.xlsx
Sheets: ['Sheet1', 'Sheet3', 'OrderTable', 'Tuesday Final', 'Tues', 'distance', 'LocationTable']
Distance sheet: distance
Tuesday sheet : Tuesday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Tuesday stores (original rows): 41
Total cube: 11537.0
Loaded cluster labels for 124 ZIPs


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Total delivery stops after splitting: 41
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 44
Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1      9     2816.0        63.0          D0 08:00          D0 14:04              08:00              14:04                0          1.57         6.08
     2     10     2337.0       308.0          D0 05:15          D0 17:57              05:15              17:57                0          7.70        12.70
     3     10     3040.0       446.0          D0 04:58          D1 07:08              04:58              31:08              600         11.15        16.15
     4      4      809.0       447.0          D0 06:34          D1 05:45              06:34              29:45              600         11.18        13.18
     5   

Wednesday

In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — WEDNESDAY (Improved)
#
# - Auto-picks Wednesday sheet (e.g., "Wednesday Final", "Wednesday", etc.)
# - Better first solution: PARALLEL_CHEAPHEAST_INSERTION
# - Adds fixed vehicle cost to reduce # of trucks
# - Optional soft cluster grouping if LocationTable has Cluster_Hierarchical
# - Output: wednesday_ortools_routes_schedule.xlsx
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

DAY = 24 * 60

# Objective shaping:
VEHICLE_FIXED_COST = 250000   # higher => fewer trucks (tune 50k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000     # set 0 to disable

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

wednesday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("wednesday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("wednesday",))
)
if wednesday_sheet is None:
    raise ValueError("Could not find a Wednesday sheet (e.g., 'Wednesday Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet :", distance_sheet)
print("Wednesday sheet:", wednesday_sheet)
print("Location sheet :", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN WEDNESDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=wednesday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Wednesday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.use_full_propagation = True
params.log_search = True
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "wednesday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Wednesday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Wednesday Final', 'distance', 'LocationTable']
Distance sheet : distance
Wednesday sheet: Wednesday Final
Location sheet : LocationTable
Distance matrix shape: (123, 123)
Wednesday stores (original rows): 39
Total cube: 15192.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 39
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 45


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1     11     2773.0       269.0          D0 05:32          D0 17:45              05:32              17:45                0          6.72        12.22
     2      3     3197.0       258.0          D0 07:51          D0 16:42              07:51              16:42                0          6.45         8.85
     3      7     3181.0       264.0          D0 07:33          D0 18:11              07:33              18:11                0          6.60        10.64
     4      6     3115.0       179.0          D0 07:14          D0 15:18              07:14              15:18                0          4.47         8.08
     5     12     2926.0       403.0          D0 05:18          D1 07:22              05:18              31:22     

Thursday

In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — THURSDAY (Improved)
#
# - Auto-picks Thursday sheet (e.g., "Thursday Final", "Thursday", etc.)
# - Better first solution: PARALLEL_CHEAPEST_INSERTION
# - Adds fixed vehicle cost to reduce # of trucks
# - Optional soft cluster grouping if LocationTable has Cluster_Hierarchical
# - Output: thursday_ortools_routes_schedule.xlsx
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

DAY = 24 * 60

# Objective shaping:
VEHICLE_FIXED_COST = 250000   # higher => fewer trucks (tune 50k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000     # set 0 to disable

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

thursday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("thursday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("thursday",))
)
if thursday_sheet is None:
    raise ValueError("Could not find a Thursday sheet (e.g., 'Thursday Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet :", distance_sheet)
print("Thursday sheet :", thursday_sheet)
print("Location sheet :", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN THURSDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=thursday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Thursday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.use_full_propagation = True
params.log_search = True
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "thursday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Thursday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Thursday Final', 'distance', 'LocationTable']
Distance sheet : distance
Thursday sheet : Thursday Final
Location sheet : LocationTable
Distance matrix shape: (123, 123)
Thursday stores (original rows): 45
Total cube: 15009.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 45
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 45


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1     11     3000.0       187.0          D0 07:44          D0 17:54              07:44              17:54                0          4.67        10.18
     2     10     3133.0       288.0          D0 05:30          D0 17:42              05:30              17:42                0          7.20        12.20
     3      5     3114.0       282.0          D0 07:32          D0 17:53              07:32              17:53                0          7.05        10.36
     4      8     2924.0       267.0          D0 07:48          D0 18:40              07:48              18:40                0          6.67        10.87
     5     11     2838.0       431.0          D0 04:58          D1 07:15              04:58              31:15     

Friday

In [ ]:
# ============================================================
# ENHANCED END-TO-END CODE (Colab / Python) — FRIDAY (Improved)
#
# - Auto-picks Friday sheet (e.g., "Friday Final", "Friday", etc.)
# - Better first solution: PARALLEL_CHEAPEST_INSERTION
# - Adds fixed vehicle cost to reduce # of trucks
# - Optional soft cluster grouping if LocationTable has Cluster_Hierarchical
# - Output: friday_ortools_routes_schedule.xlsx
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== SETTINGS ====================
DC = "01887"
CAPACITY   = 3200

OPEN       = 8  * 60
CLOSE      = 18 * 60

MAX_DRIVE  = 11 * 60
MAX_DUTY   = 14 * 60
REST_MIN   = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

DAY = 24 * 60

# Objective shaping:
VEHICLE_FIXED_COST = 250000   # higher => fewer trucks (tune 50k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000     # set 0 to disable

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1
        mm = 0
    return f"{hh:02d}:{mm:02d}"

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

friday_sheet = (
    pick_sheet(xl.sheet_names, contains_all=("friday", "final"))
    or pick_sheet(xl.sheet_names, contains_all=("friday",))
)
if friday_sheet is None:
    raise ValueError("Could not find a Friday sheet (e.g., 'Friday Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet:", distance_sheet)
print("Friday sheet  :", friday_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common  = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN FRIDAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=friday_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"Friday stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

# ==================== 7) OR-TOOLS VRP ====================

min_trailers  = math.ceil(sum(demands) / CAPACITY)
num_vehicles  = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

def cost_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

    cl_pen = 0
    if i != 0 and j != 0 and clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
        cl_pen = CLUSTER_PENALTY
    return dist_cost + cl_pen

cost_idx = routing.RegisterTransitCallback(cost_cb)
routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

for v in range(num_vehicles):
    routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

def demand_cb(fi):
    return int(round(demands[manager.IndexToNode(fi)]))

routing.AddDimensionWithVehicleCapacity(
    routing.RegisterUnaryTransitCallback(demand_cb),
    0,
    [CAPACITY] * num_vehicles,
    True,
    "Capacity"
)

def time_cb(fi, ti):
    i = manager.IndexToNode(fi)
    j = manager.IndexToNode(ti)
    travel = travel_min[i][j]
    svc = service_times[i] if i != 0 else 0.0
    return int(round(travel + svc))

time_idx = routing.RegisterTransitCallback(time_cb)
routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
time_dim = routing.GetDimensionOrDie("Time")

for node in range(1, n):
    latest = int(CLOSE - math.ceil(service_times[node]))
    if latest < OPEN:
        raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
    idx = manager.NodeToIndex(node)
    time_dim.CumulVar(idx).SetRange(OPEN, latest)

for v in range(num_vehicles):
    time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.use_full_propagation = True
params.log_search = True
params.time_limit.FromSeconds(SOLVER_SECONDS)

solution = routing.SolveWithParameters(params)
if not solution:
    raise RuntimeError(
        "OR-Tools found no feasible solution.\n"
        "Try increasing SOLVER_SECONDS, lowering CLUSTER_PENALTY, or check input data."
    )

routes = []
for v in range(num_vehicles):
    idx = routing.Start(v)
    nxt = solution.Value(routing.NextVar(idx))
    if routing.IsEnd(nxt):
        continue
    route = []
    while not routing.IsEnd(idx):
        route.append(manager.IndexToNode(idx))
        idx = solution.Value(routing.NextVar(idx))
    route.append(0)
    routes.append(route)

print(f"Routes from OR-Tools (before HOS split): {len(routes)}")

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"Routes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    if len(route) <= 2:
        continue

    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = "friday_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Friday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Friday Final', 'distance', 'LocationTable']
Distance sheet: distance
Friday sheet  : Friday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Friday stores (original rows): 41
Total cube: 13468.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 42
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 45


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Routes from OR-Tools (before HOS split): 5
Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  RouteMiles Dispatch(DayTime) ReturnDC(DayTime) Dispatch(HH:MM>24) ReturnDC(HH:MM>24)  RestInsertedMin  TotalDriveHr  TotalDutyHr
     1      5     1949.0       227.0          D0 07:48          D0 15:58              07:48              15:58                0          5.67         8.18
     2     11     3119.0       442.0          D0 04:58          D1 07:32              04:58              31:32              600         11.05        16.55
     3     13     2564.0       137.0          D0 06:51          D0 17:55              06:51              17:55                0          3.42        11.07
     4      1     3200.0        44.0          D0 07:27          D0 10:11              07:27              10:11                0          1.10         2.74
     5     12     2636.0       310.0          D0 04:52          D0 18:38              04:52              18:38     

check for optimal soln

In [ ]:
# ============================================================
# FULL, ENHANCED END-TO-END CODE (Colab / Python) — ANY DAY
# (Configured for FRIDAY by default; change DAY_NAME)
#
# WORKS ON YOUR OR-TOOLS VERSION:
# - No params.random_seed
# - No routing.solver().SetRandSeed
#
# “Near-optimality” CHECK:
# - Run multiple diversified runs (different first_solution_strategy)
# - Keep best objective
#
# PRINTS:
# - ObjectiveValue + arc/fixed breakdown
# - route miles per route + total miles
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== USER SETTINGS ====================
DAY_NAME = "friday"   # "monday" / "tuesday" / "wednesday" / "thursday" / "friday"

DC = "01887"
CAPACITY = 3200

OPEN  = 8 * 60
CLOSE = 18 * 60

MAX_DRIVE = 11 * 60
MAX_DUTY  = 14 * 60
REST_MIN  = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

# Objective shaping:
VEHICLE_FIXED_COST = 250000     # higher => fewer trucks (try 100k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000       # set 0 to disable

DAY = 24 * 60

# How many independent runs to compare (no seeding available)
N_RUNS = 5

# Diversify first-solution across runs to explore different regions
FIRST_SOLUTIONS = [
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION,
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC,
    routing_enums_pb2.FirstSolutionStrategy.SAVINGS,
    routing_enums_pb2.FirstSolutionStrategy.CHRISTOFIDES,
    routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC,
]

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"{hh:02d}:{mm:02d}"

def route_miles(route_nodes, dist_miles):
    m = 0.0
    for a, b in zip(route_nodes, route_nodes[1:]):
        m += dist_miles[a][b]
    return m

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

day_sheet = (
    pick_sheet(xl.sheet_names, contains_all=(DAY_NAME, "final"))
    or pick_sheet(xl.sheet_names, contains_all=(DAY_NAME,))
)
if day_sheet is None:
    raise ValueError(f"Could not find a sheet for {DAY_NAME} (e.g., '{DAY_NAME.title()} Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet:", distance_sheet)
print("Day sheet     :", day_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN DAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=day_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"{DAY_NAME.title()} stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

min_trailers = math.ceil(sum(demands) / CAPACITY)
num_vehicles = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

# ==================== SOLVE ONE RUN ====================

def solve_one(first_solution, verbose_log=False):
    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def cost_cb(fi, ti):
        i = manager.IndexToNode(fi)
        j = manager.IndexToNode(ti)
        dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

        cl_pen = 0
        if CLUSTER_PENALTY and i != 0 and j != 0:
            if clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
                cl_pen = CLUSTER_PENALTY
        return dist_cost + cl_pen

    cost_idx = routing.RegisterTransitCallback(cost_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

    for v in range(num_vehicles):
        routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

    def demand_cb(fi):
        return int(round(demands[manager.IndexToNode(fi)]))

    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(demand_cb),
        0,
        [CAPACITY] * num_vehicles,
        True,
        "Capacity"
    )

    def time_cb(fi, ti):
        i = manager.IndexToNode(fi)
        j = manager.IndexToNode(ti)
        travel = travel_min[i][j]
        svc = service_times[i] if i != 0 else 0.0
        return int(round(travel + svc))

    time_idx = routing.RegisterTransitCallback(time_cb)
    routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
    time_dim = routing.GetDimensionOrDie("Time")

    for node in range(1, n):
        latest = int(CLOSE - math.ceil(service_times[node]))
        if latest < OPEN:
            raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
        idx = manager.NodeToIndex(node)
        time_dim.CumulVar(idx).SetRange(OPEN, latest)

    for v in range(num_vehicles):
        time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = first_solution
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.use_full_propagation = True
    params.time_limit.FromSeconds(SOLVER_SECONDS)
    params.log_search = bool(verbose_log)

    sol = routing.SolveWithParameters(params)
    if not sol:
        return None

    # Extract routes + used vehicle count
    routes = []
    used_vehicles = 0
    for v in range(num_vehicles):
        idx = routing.Start(v)
        nxt = sol.Value(routing.NextVar(idx))
        if routing.IsEnd(nxt):
            continue
        used_vehicles += 1
        route = []
        while not routing.IsEnd(idx):
            route.append(manager.IndexToNode(idx))
            idx = sol.Value(routing.NextVar(idx))
        route.append(0)
        routes.append(route)

    # Arc cost total
    arc_cost_total = 0
    for v in range(num_vehicles):
        idx = routing.Start(v)
        if routing.IsEnd(sol.Value(routing.NextVar(idx))):
            continue
        while not routing.IsEnd(idx):
            nxt = sol.Value(routing.NextVar(idx))
            arc_cost_total += routing.GetArcCostForVehicle(idx, nxt, v)
            idx = nxt

    fixed_cost_total = used_vehicles * VEHICLE_FIXED_COST
    obj = sol.ObjectiveValue()

    total_mi = sum(route_miles(r, dist_miles) for r in routes)

    return {
        "objective": obj,
        "arc_cost": arc_cost_total,
        "fixed_cost": fixed_cost_total,
        "used_vehicles": used_vehicles,
        "routes": routes,
        "total_miles": total_mi,
        "first_solution": first_solution,
    }

# ==================== 7) MULTI-RUN “NEAR-OPTIMALITY” CHECK ====================

results = []
for run_i in range(N_RUNS):
    fs = FIRST_SOLUTIONS[run_i % len(FIRST_SOLUTIONS)]
    verbose = (run_i == 0)
    print(f"\n--- Run {run_i+1}/{N_RUNS} | first_solution={fs} | log_search={verbose} ---")
    res = solve_one(fs, verbose_log=verbose)
    if res is None:
        print("No solution in this run.")
        continue
    print(f"Run {run_i+1}: objective={res['objective']} trucks={res['used_vehicles']} miles={res['total_miles']:.1f}")
    results.append(res)

if not results:
    raise RuntimeError("No feasible solution found. Check constraints/input data.")

best = min(results, key=lambda r: r["objective"])
routes = best["routes"]

print("\n==================== BEST RUN ====================")
print("Final objective:", best["objective"])
print("Used vehicles:", best["used_vehicles"])
print("Arc cost total:", best["arc_cost"])
print("Fixed cost total:", best["fixed_cost"])
print("Arc+Fixed:", best["arc_cost"] + best["fixed_cost"])
print("Objective - (Arc+Fixed):", best["objective"] - (best["arc_cost"] + best["fixed_cost"]))
print("Total route miles (pre-HOS):", round(best["total_miles"], 1))
print("==================================================")

print("\n--- Route miles (pre-HOS) ---")
for r_i, r in enumerate(routes, start=1):
    print(f"Route {r_i:02d}: stops={len(r)-2:2d}, miles={route_miles(r, dist_miles):.1f}")
print("Total miles:", round(best["total_miles"], 1))

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"\nRoutes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
        "ObjectiveValue": best["objective"],
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = f"{DAY_NAME}_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")


Using file: solution deliveries Friday.xlsx
Sheets: ['Sheet1', 'Sheet2', 'OrderTable', 'Friday Final', 'distance', 'LocationTable']
Distance sheet: distance
Day sheet     : Friday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Friday stores (original rows): 41
Total cube: 13468.0
Loaded cluster labels for 123 ZIPs
Total delivery stops after splitting: 42
Min trailers (capacity only): 5
Vehicles given to OR-Tools  : 45

--- Run 1/5 | first_solution=8 | log_search=True ---


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Run 1: objective=2418000 trucks=5 miles=1160.0

--- Run 2/5 | first_solution=3 | log_search=False ---
Run 2: objective=2418000 trucks=5 miles=1160.0

--- Run 3/5 | first_solution=10 | log_search=False ---
Run 3: objective=2418000 trucks=5 miles=1160.0

--- Run 4/5 | first_solution=13 | log_search=False ---
Run 4: objective=2418000 trucks=5 miles=1160.0

--- Run 5/5 | first_solution=15 | log_search=False ---
Run 5: objective=2418000 trucks=5 miles=1160.0

==================== BEST RUN ====================
Final objective: 2418000
Used vehicles: 5
Arc cost total: 2418000
Fixed cost total: 1250000
Arc+Fixed: 3668000
Objective - (Arc+Fixed): -1250000
Total route miles (pre-HOS): 1160.0

--- Route miles (pre-HOS) ---
Route 01: stops= 5, miles=227.0
Route 02: stops=11, miles=442.0
Route 03: stops=13, miles=137.0
Route 04: stops= 1, miles=44.0
Route 05: stops=12, miles=310.0
Total miles: 1160.0

Routes after HOS enforcement: 5

--- SUMMARY (first 10 trucks) ---
 Truck  Stops  TotalCube  Route

In [ ]:
# ============================================================
# FULL, ENHANCED END-TO-END CODE (Colab / Python) — TUESDAY
# (Same as your script; only set DAY_NAME="tuesday")
# ============================================================

import glob
import math
import pandas as pd
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

# ==================== USER SETTINGS ====================
DAY_NAME = "tuesday"  # <-- changed to TUESDAY

DC = "01887"
CAPACITY = 3200

OPEN  = 8 * 60
CLOSE = 18 * 60

MAX_DRIVE = 11 * 60
MAX_DUTY  = 14 * 60
REST_MIN  = 10 * 60

SOLVER_SECONDS = 180
EXTRA_VEHICLES = 40

# Objective shaping:
VEHICLE_FIXED_COST = 250000     # higher => fewer trucks (try 100k–500k)
DIST_COST_SCALE    = 1000
CLUSTER_PENALTY    = 2000       # set 0 to disable

DAY = 24 * 60

# How many independent runs to compare (no seeding available)
N_RUNS = 5

# Diversify first-solution across runs to explore different regions
FIRST_SOLUTIONS = [
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION,
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC,
    routing_enums_pb2.FirstSolutionStrategy.SAVINGS,
    routing_enums_pb2.FirstSolutionStrategy.CHRISTOFIDES,
    routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC,
]

# ==================== HELPERS ====================

def pick_sheet(candidates, contains_all=()):
    keys = [k.lower() for k in contains_all]
    for s in candidates:
        if all(k in s.lower() for k in keys):
            return s
    return None

def fmt_day_time(total_min):
    m = float(total_min)
    d = int(m // DAY)
    tod = m - d * DAY
    hh, mm = divmod(int(round(tod)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"D{d} {hh:02d}:{mm:02d}"

def fmt_hhmm(total_min):
    m = float(total_min)
    hh, mm = divmod(int(round(m)), 60)
    if mm == 60:
        hh += 1; mm = 0
    return f"{hh:02d}:{mm:02d}"

def route_miles(route_nodes, dist_miles):
    m = 0.0
    for a, b in zip(route_nodes, route_nodes[1:]):
        m += dist_miles[a][b]
    return m

# ==================== 1) LOCATE EXCEL ====================

xlsx_files = sorted(glob.glob("*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError("No .xlsx file found. Upload your Excel first.")
file_path = xlsx_files[0]
print("Using file:", file_path)

xl = pd.ExcelFile(file_path)
print("Sheets:", xl.sheet_names)

# ==================== 2) PICK SHEETS ====================

distance_sheet = pick_sheet(xl.sheet_names, contains_all=("distance",)) or "distance"

day_sheet = (
    pick_sheet(xl.sheet_names, contains_all=(DAY_NAME, "final"))
    or pick_sheet(xl.sheet_names, contains_all=(DAY_NAME,))
)
if day_sheet is None:
    raise ValueError(f"Could not find a sheet for {DAY_NAME} (e.g., '{DAY_NAME.title()} Final').")

location_sheet = pick_sheet(xl.sheet_names, contains_all=("location",))  # e.g., LocationTable

print("Distance sheet:", distance_sheet)
print("Day sheet     :", day_sheet)
print("Location sheet:", location_sheet)

# ==================== 3) READ & CLEAN DISTANCE MATRIX ====================

raw = pd.read_excel(file_path, sheet_name=distance_sheet)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "Zip"})
raw = raw[raw["Zip"].astype(str).str.strip().str.lower() != "zip"]
raw = raw.drop(columns=[c for c in raw.columns if str(c).strip().lower() == "id"], errors="ignore")
raw = raw.dropna(axis=1, how="all")

raw["Zip"] = raw["Zip"].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
raw = raw.set_index("Zip")
raw.columns = (
    pd.Series(raw.columns).astype(str).str.strip()
      .str.extract(r"(\d+)")[0].astype(str).str.zfill(5).tolist()
)

dist_df = raw.apply(pd.to_numeric, errors="coerce")
common = dist_df.index.intersection(dist_df.columns)
dist_df = dist_df.loc[common, common]
print("Distance matrix shape:", dist_df.shape)

if dist_df.isna().any().any():
    bad = dist_df.isna().stack()
    bad = bad[bad].index.tolist()[:15]
    raise ValueError(f"Distance matrix has NaNs: {bad}")

# ==================== 4) READ & CLEAN DAY DATA ====================

day_df = pd.read_excel(file_path, sheet_name=day_sheet)

zip_col = next((c for c in ["Row Labels","TOZIP","ToZip","tozip","Zip","ZIP"] if c in day_df.columns), None)
cube_col = next((c for c in ["Sum of Cube","Sum of CUBE","sum of cube","CUBE","Cube","sum_cube","SumCube"] if c in day_df.columns), None)
if zip_col is None or cube_col is None:
    raise ValueError(f"Couldn't find ZIP/Cube columns. Found: {day_df.columns.tolist()}")

day_df[zip_col]  = day_df[zip_col].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
day_df[cube_col] = pd.to_numeric(day_df[cube_col], errors="coerce").fillna(0.0)
day_df = day_df[day_df[zip_col].notna()].copy()

unload_col = next((c for c in ["Unloading Time","UnloadingTime","Unload","Unload Time","ServiceMin"] if c in day_df.columns), None)

print(f"{DAY_NAME.title()} stores (original rows): {len(day_df)}")
print(f"Total cube: {float(day_df[cube_col].sum()):.1f}")

# ==================== 4.5) OPTIONAL CLUSTER MAP ====================

zip_to_cluster = {}
if location_sheet:
    loc = pd.read_excel(file_path, sheet_name=location_sheet)
    loc.columns = [str(c).strip() for c in loc.columns]
    zc = next((c for c in ["ZIP","Zip","zip"] if c in loc.columns), None)
    cc = next((c for c in ["Cluster_Hierarchical","Cluster","cluster"] if c in loc.columns), None)
    if zc and cc:
        tmp = loc[[zc, cc]].copy()
        tmp[zc] = tmp[zc].astype(str).str.extract(r"(\d+)")[0].astype(str).str.zfill(5)
        zip_to_cluster = dict(zip(tmp[zc], tmp[cc].astype(str)))
        print("Loaded cluster labels for", len(zip_to_cluster), "ZIPs")
    else:
        print("LocationTable found, but cluster columns not found — skipping cluster preference.")

# ==================== 5) SPLIT STOPS EXCEEDING CAPACITY ====================

split_rows = []
for _, r in day_df.iterrows():
    z = r[zip_col]
    cube = float(r[cube_col])
    if cube <= 0:
        continue

    unload_val = None
    if unload_col:
        try:
            unload_val = float(r[unload_col])
        except Exception:
            pass

    k = math.ceil(cube / CAPACITY)
    remaining = cube
    for part in range(1, k + 1):
        chunk = min(CAPACITY, remaining)
        remaining -= chunk
        split_rows.append({
            "BaseZIP": z,
            "VisitID": f"{z}#{part}" if k > 1 else z,
            "Cube": chunk,
            "UnloadFromSheet": unload_val,
            "Cluster": zip_to_cluster.get(z, "NA"),
        })

split_df = pd.DataFrame(split_rows)
print(f"Total delivery stops after splitting: {len(split_df)}")

# ==================== 6) BUILD NODE ARRAYS ====================

nodes_id = [DC] + split_df["VisitID"].tolist()
base_zip = [DC] + split_df["BaseZIP"].tolist()
clusters = ["DC"] + split_df["Cluster"].tolist()

missing = [z for z in set(base_zip) if z not in dist_df.index]
if missing:
    raise ValueError(f"ZIPs missing from distance matrix: {missing[:15]}")

n = len(base_zip)

dist_miles = [[float(dist_df.loc[base_zip[i], base_zip[j]]) for j in range(n)] for i in range(n)]
travel_min = [[dist_miles[i][j] * 1.5 for j in range(n)] for i in range(n)]  # 40mph => 1.5 min/mi

demands = [0] + split_df["Cube"].astype(float).tolist()

service_times = [0.0]
for _, r in split_df.iterrows():
    if unload_col and pd.notna(r["UnloadFromSheet"]):
        svc = max(0.0, float(r["UnloadFromSheet"]))
    else:
        svc = max(30.0, 0.030 * float(r["Cube"]))
    service_times.append(svc)

min_trailers = math.ceil(sum(demands) / CAPACITY)
num_vehicles = min_trailers + EXTRA_VEHICLES
print(f"Min trailers (capacity only): {min_trailers}")
print(f"Vehicles given to OR-Tools  : {num_vehicles}")

# ==================== SOLVE ONE RUN ====================

def solve_one(first_solution, verbose_log=False):
    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def cost_cb(fi, ti):
        i = manager.IndexToNode(fi)
        j = manager.IndexToNode(ti)
        dist_cost = int(dist_miles[i][j] * DIST_COST_SCALE)

        cl_pen = 0
        if CLUSTER_PENALTY and i != 0 and j != 0:
            if clusters[i] != clusters[j] and clusters[i] != "NA" and clusters[j] != "NA":
                cl_pen = CLUSTER_PENALTY
        return dist_cost + cl_pen

    cost_idx = routing.RegisterTransitCallback(cost_cb)
    routing.SetArcCostEvaluatorOfAllVehicles(cost_idx)

    for v in range(num_vehicles):
        routing.SetFixedCostOfVehicle(VEHICLE_FIXED_COST, v)

    def demand_cb(fi):
        return int(round(demands[manager.IndexToNode(fi)]))

    routing.AddDimensionWithVehicleCapacity(
        routing.RegisterUnaryTransitCallback(demand_cb),
        0,
        [CAPACITY] * num_vehicles,
        True,
        "Capacity"
    )

    def time_cb(fi, ti):
        i = manager.IndexToNode(fi)
        j = manager.IndexToNode(ti)
        travel = travel_min[i][j]
        svc = service_times[i] if i != 0 else 0.0
        return int(round(travel + svc))

    time_idx = routing.RegisterTransitCallback(time_cb)
    routing.AddDimension(time_idx, 24*60, 24*60, False, "Time")
    time_dim = routing.GetDimensionOrDie("Time")

    for node in range(1, n):
        latest = int(CLOSE - math.ceil(service_times[node]))
        if latest < OPEN:
            raise ValueError(f"Node {nodes_id[node]}: service_time={service_times[node]:.0f} too large for 08–18.")
        idx = manager.NodeToIndex(node)
        time_dim.CumulVar(idx).SetRange(OPEN, latest)

    for v in range(num_vehicles):
        time_dim.CumulVar(routing.Start(v)).SetRange(0, OPEN)

    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = first_solution
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.use_full_propagation = True
    params.time_limit.FromSeconds(SOLVER_SECONDS)
    params.log_search = bool(verbose_log)

    sol = routing.SolveWithParameters(params)
    if not sol:
        return None

    routes = []
    used_vehicles = 0
    for v in range(num_vehicles):
        idx = routing.Start(v)
        nxt = sol.Value(routing.NextVar(idx))
        if routing.IsEnd(nxt):
            continue
        used_vehicles += 1
        route = []
        while not routing.IsEnd(idx):
            route.append(manager.IndexToNode(idx))
            idx = sol.Value(routing.NextVar(idx))
        route.append(0)
        routes.append(route)

    arc_cost_total = 0
    for v in range(num_vehicles):
        idx = routing.Start(v)
        if routing.IsEnd(sol.Value(routing.NextVar(idx))):
            continue
        while not routing.IsEnd(idx):
            nxt = sol.Value(routing.NextVar(idx))
            arc_cost_total += routing.GetArcCostForVehicle(idx, nxt, v)
            idx = nxt

    fixed_cost_total = used_vehicles * VEHICLE_FIXED_COST
    obj = sol.ObjectiveValue()
    total_mi = sum(route_miles(r, dist_miles) for r in routes)

    return {
        "objective": obj,
        "arc_cost": arc_cost_total,
        "fixed_cost": fixed_cost_total,
        "used_vehicles": used_vehicles,
        "routes": routes,
        "total_miles": total_mi,
        "first_solution": first_solution,
    }

# ==================== 7) MULTI-RUN “NEAR-OPTIMALITY” CHECK ====================

results = []
for run_i in range(N_RUNS):
    fs = FIRST_SOLUTIONS[run_i % len(FIRST_SOLUTIONS)]
    verbose = (run_i == 0)
    print(f"\n--- Run {run_i+1}/{N_RUNS} | first_solution={fs} | log_search={verbose} ---")
    res = solve_one(fs, verbose_log=verbose)
    if res is None:
        print("No solution in this run.")
        continue
    print(f"Run {run_i+1}: objective={res['objective']} trucks={res['used_vehicles']} miles={res['total_miles']:.1f}")
    results.append(res)

if not results:
    raise RuntimeError("No feasible solution found. Check constraints/input data.")

best = min(results, key=lambda r: r["objective"])
routes = best["routes"]

print("\n==================== BEST RUN ====================")
print("Final objective:", best["objective"])
print("Used vehicles:", best["used_vehicles"])
print("Arc cost total:", best["arc_cost"])
print("Fixed cost total:", best["fixed_cost"])
print("Arc+Fixed:", best["arc_cost"] + best["fixed_cost"])
print("Objective - (Arc+Fixed):", best["objective"] - (best["arc_cost"] + best["fixed_cost"]))
print("Total route miles (pre-HOS):", round(best["total_miles"], 1))
print("==================================================")

print("\n--- Route miles (pre-HOS) ---")
for r_i, r in enumerate(routes, start=1):
    print(f"Route {r_i:02d}: stops={len(r)-2:2d}, miles={route_miles(r, dist_miles):.1f}")
print("Total miles:", round(best["total_miles"], 1))

# ==================== 8) ENFORCE HOS BY SPLITTING ROUTES ====================

def split_for_hos(route):
    if len(route) <= 2:
        return [route]

    first = route[1]
    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty = 0.0
    prev = 0

    for k, cur in enumerate(route[1:-1], start=1):
        leg_travel = travel_min[prev][cur]

        if drive + leg_travel > MAX_DRIVE or duty + leg_travel > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        t += leg_travel
        drive += leg_travel
        duty  += leg_travel

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        svc = service_times[cur]
        if t + svc > CLOSE + 1e-6:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        duty += svc
        t += svc

        if drive > MAX_DRIVE or duty > MAX_DUTY:
            left  = route[:k] + [0]
            right = [0] + route[k:]
            return split_for_hos(left) + split_for_hos(right)

        prev = cur

    return [route]

hos_routes = []
for r in routes:
    hos_routes.extend(split_for_hos(r))

routes = [r for r in hos_routes if len(r) > 2]
print(f"\nRoutes after HOS enforcement: {len(routes)}")

# ==================== 9) BUILD SCHEDULE ====================

schedule_rows = []
summary_rows  = []
truck_num     = 0

for route in routes:
    truck_num += 1
    first = route[1]

    dispatch = OPEN - travel_min[0][first]
    t = dispatch
    drive = 0.0
    duty  = 0.0
    miles = 0.0
    prev  = 0
    stop_n = 0

    for cur in route[1:-1]:
        leg_t  = travel_min[prev][cur]
        leg_mi = dist_miles[prev][cur]

        t += leg_t
        drive += leg_t
        duty  += leg_t
        miles += leg_mi

        if t < OPEN:
            duty += (OPEN - t)
            t = OPEN

        arrival = t
        svc = service_times[cur]
        depart = arrival + svc

        duty += svc
        t = depart
        stop_n += 1

        schedule_rows.append({
            "Truck": truck_num,
            "Stop#": stop_n,
            "VisitID": nodes_id[cur],
            "StoreZIP": base_zip[cur],
            "Cluster": clusters[cur],
            "Cube": round(demands[cur], 1),
            "ServiceMin": round(svc, 1),
            "Arrival(DayTime)": fmt_day_time(arrival),
            "Depart(DayTime)" : fmt_day_time(depart),
            "DriveHrSoFar": round(drive / 60, 2),
            "DutyHrSoFar" : round(duty  / 60, 2),
        })
        prev = cur

    ret_t  = travel_min[prev][0]
    ret_mi = dist_miles[prev][0]

    rest_inserted = 0.0
    if drive + ret_t > MAX_DRIVE or duty + ret_t > MAX_DUTY:
        rest_inserted = REST_MIN
        t += REST_MIN

    t += ret_t
    miles += ret_mi

    summary_rows.append({
        "Truck": truck_num,
        "Stops": stop_n,
        "TotalCube": round(sum(demands[nd] for nd in route if nd != 0), 1),
        "RouteMiles": round(miles, 1),
        "Dispatch(DayTime)": fmt_day_time(dispatch),
        "ReturnDC(DayTime)": fmt_day_time(t),
        "Dispatch(HH:MM>24)": fmt_hhmm(dispatch),
        "ReturnDC(HH:MM>24)": fmt_hhmm(t),
        "RestInsertedMin": int(rest_inserted),
        "TotalDriveHr": round((drive + ret_t) / 60, 2),
        "TotalDutyHr" : round((duty  + ret_t) / 60, 2),
        "ObjectiveValue": best["objective"],
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values(["Truck","Stop#"]).reset_index(drop=True)
summary_df  = pd.DataFrame(summary_rows ).sort_values("Truck").reset_index(drop=True)

print("\n--- SUMMARY (first 10 trucks) ---")
print(summary_df.head(10).to_string(index=False))

truck_plan_df = schedule_df[[
    "Truck","Stop#","StoreZIP","Arrival(DayTime)","Depart(DayTime)",
    "Cube","ServiceMin","VisitID","Cluster"
]].copy()
truck_plan_df["Window"] = truck_plan_df["Arrival(DayTime)"] + " → " + truck_plan_df["Depart(DayTime)"]

# ==================== 10) SAVE OUTPUT ====================

out_path = f"{DAY_NAME}_ortools_routes_schedule.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as w:
    summary_df.to_excel(w,    sheet_name="Summary",   index=False)
    schedule_df.to_excel(w,   sheet_name="Schedule",  index=False)
    truck_plan_df.to_excel(w, sheet_name="TruckPlan", index=False)

print(f"\nSaved: {out_path}")
print("Time format: 'D0 HH:MM' = same day, 'D1 HH:MM' = next day")
print("HH:MM>24 column: e.g., 31:32 = 07:32 next day")

Using file: solution deliveries Tuesday.xlsx
Sheets: ['Sheet1', 'Sheet3', 'OrderTable', 'Tuesday Final', 'Tues', 'distance', 'LocationTable']
Distance sheet: distance
Day sheet     : Tuesday Final
Location sheet: LocationTable
Distance matrix shape: (123, 123)
Tuesday stores (original rows): 41
Total cube: 11537.0


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Loaded cluster labels for 124 ZIPs
Total delivery stops after splitting: 41
Min trailers (capacity only): 4
Vehicles given to OR-Tools  : 44

--- Run 1/5 | first_solution=8 | log_search=True ---
Run 1: objective=2605000 trucks=5 miles=1345.0

--- Run 2/5 | first_solution=3 | log_search=False ---
Run 2: objective=2605000 trucks=5 miles=1345.0

--- Run 3/5 | first_solution=10 | log_search=False ---
Run 3: objective=2605000 trucks=5 miles=1345.0

--- Run 4/5 | first_solution=13 | log_search=False ---
Run 4: objective=2605000 trucks=5 miles=1345.0

--- Run 5/5 | first_solution=15 | log_search=False ---
Run 5: objective=2605000 trucks=5 miles=1345.0

==================== BEST RUN ====================
Final objective: 2605000
Used vehicles: 5
Arc cost total: 2605000
Fixed cost total: 1250000
Arc+Fixed: 3855000
Objective - (Arc+Fixed): -1250000
Total route miles (pre-HOS): 1345.0

--- Route miles (pre-HOS) ---
Route 01: stops= 9, miles=63.0
Route 02: stops=10, miles=308.0
Route 03: stops=10, 